# B2B SaaS — Sales & Customer Health Analysis

**[github.com/dkamehat](https://github.com/dkamehat)**

This notebook applies a four-dashboard analytical framework to a synthetic B2B SaaS business. Schema mirrors Salesforce standard objects (Accounts, Opportunities, Activities, Users) extended with Customer Success tables (Subscriptions, UsageMetrics, HealthScores).

All data is synthetic. No real customer information appears anywhere.

---

## How to read this notebook

Each of the four analyses follows the same pattern:

1. **The question** — what decision does this section support?
2. **The SQL** — runnable on DuckDB against the flat-file CSVs
3. **The chart** — interactive Plotly visualization
4. **Design Notes** — what trade-offs were made and why

If short on time: read the question and Design Notes for each section. That's the analytical reasoning; the charts are the evidence.

---

## Contents

- §1. Sales Performance Overview — Exec / CRO view
- §2. Pipeline Health — Sales Manager view
- §3. Account Insights — PM / GTM Strategy view
- §4. Customer Health — CS Manager view


## Setup

In [1]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

con = duckdb.connect(database=':memory:')

for name in ['Accounts', 'Opportunities', 'Activities', 'Users',
            'Subscriptions', 'UsageMetrics', 'HealthScores']:
    con.execute(f"CREATE VIEW {name} AS SELECT * FROM read_csv_auto('../data/{name}.csv')")

row_counts = con.execute("""
SELECT 'Accounts'      AS table_name, COUNT(*) AS row_count FROM Accounts
UNION ALL SELECT 'Opportunities',  COUNT(*) FROM Opportunities
UNION ALL SELECT 'Activities',     COUNT(*) FROM Activities
UNION ALL SELECT 'Users',          COUNT(*) FROM Users
UNION ALL SELECT 'Subscriptions',  COUNT(*) FROM Subscriptions
UNION ALL SELECT 'UsageMetrics',   COUNT(*) FROM UsageMetrics
UNION ALL SELECT 'HealthScores',   COUNT(*) FROM HealthScores
""").df()
row_counts

,table_name,row_count
0,Accounts,800
1,Opportunities,1342
2,Activities,12215
3,Users,12
4,Subscriptions,331
5,UsageMetrics,3559
6,HealthScores,331


In [2]:
PRIMARY = '#2E5BFF'
ACCENT  = '#00C2A8'
DANGER  = '#E94E1B'
NEUTRAL = '#1A1F36'

import plotly.io as pio
template = go.layout.Template()
template.layout = go.Layout(
    font=dict(family='Inter, system-ui, sans-serif', size=13, color=NEUTRAL),
    plot_bgcolor='white',
    paper_bgcolor='white',
    colorway=[PRIMARY, ACCENT, DANGER, '#6E6759', '#cdc6b9'],
    xaxis=dict(gridcolor='#eef0f3', zerolinecolor='#eef0f3'),
    yaxis=dict(gridcolor='#eef0f3', zerolinecolor='#eef0f3'),
    margin=dict(l=60, r=30, t=60, b=50),
)
pio.templates['portfolio'] = template
pio.templates.default = 'portfolio'

---

# §1. Sales Performance Overview

**The question:** *Are we on track this quarter, and where is the variance coming from?*

**Audience:** Executive / CRO

This section builds the leading view a sales leader opens first thing Monday morning. The lagging outputs (Won Revenue, Win Rate) sit at the top because those are the numbers the CRO is measured on. The Region × Quarter heatmap separates markets that compound from markets that spike-and-fade.

I deliberately did **not** include forecast in this view — forecast belongs on Pipeline Health, where a manager can act on it.


### 1.1 KPI summary

In [3]:
kpi = con.execute("""
WITH closed AS (
    SELECT
        Amount,
        CreatedDate,
        CloseDate,
        CASE WHEN Stage = '06 - Closed Won'  THEN 1 ELSE 0 END AS is_won,
        CASE WHEN Stage = '07 - Closed Lost' THEN 1 ELSE 0 END AS is_lost
    FROM Opportunities
    WHERE Stage IN ('06 - Closed Won', '07 - Closed Lost')
)
SELECT
    SUM(CASE WHEN is_won = 1 THEN Amount ELSE 0 END) AS won_revenue,
    SUM(is_won) * 1.0 / NULLIF(SUM(is_won + is_lost), 0) AS win_rate,
    AVG(CASE WHEN is_won = 1 THEN Amount END) AS avg_deal_size,
    AVG(CASE WHEN is_won = 1 THEN DATE_DIFF('day', CreatedDate, CloseDate) END) AS avg_cycle_days
FROM closed
""").df()

print(f"  Won Revenue:    ${kpi.iloc[0]['won_revenue']:>14,.0f}")
print(f"  Win Rate:       {kpi.iloc[0]['win_rate']*100:>14.1f}%")
print(f"  Avg Deal Size:  ${kpi.iloc[0]['avg_deal_size']:>14,.0f}")
print(f"  Avg Cycle:      {kpi.iloc[0]['avg_cycle_days']:>14.1f} days")

  Won Revenue:    $    43,169,900
  Win Rate:                 58.4%
  Avg Deal Size:  $       110,692
  Avg Cycle:               100.4 days


### 1.2 Revenue trend by month

In [4]:
trend = con.execute("""
SELECT
    DATE_TRUNC('month', CloseDate)::DATE AS month,
    SUM(Amount) FILTER (WHERE Stage = '06 - Closed Won') AS won_revenue,
    COUNT(*)    FILTER (WHERE Stage = '06 - Closed Won') AS won_count
FROM Opportunities
WHERE CloseDate >= '2025-01-01' AND CloseDate < '2026-06-01'
GROUP BY 1
ORDER BY 1
""").df()

fig = px.line(trend, x='month', y='won_revenue', markers=True, height=420,
              title='Monthly Won Revenue',
              labels={'month': 'Month', 'won_revenue': 'Won Revenue (USD)'})
fig.update_traces(line=dict(width=3))
fig.update_layout(hovermode='x unified')
fig.show()

### 1.3 Region × Quarter heatmap

In [5]:
region_q = con.execute("""
SELECT
    a.Region AS region,
    DATE_TRUNC('quarter', o.CloseDate)::DATE AS quarter,
    SUM(o.Amount) FILTER (WHERE o.Stage = '06 - Closed Won') AS won_revenue
FROM Opportunities o
JOIN Accounts a USING (AccountId)
WHERE o.CloseDate >= '2025-01-01' AND o.CloseDate < '2026-06-01'
GROUP BY 1, 2
ORDER BY 1, 2
""").df()

pivot = region_q.pivot(index='region', columns='quarter', values='won_revenue').fillna(0)
fig = px.imshow(pivot, color_continuous_scale='Blues', aspect='auto', height=400,
                labels=dict(color='Won Revenue'),
                title='Won Revenue by Region × Quarter')
fig.update_xaxes(title='Quarter'); fig.update_yaxes(title='Region')
fig.show()

### 1.4 Industry × Segment treemap

In [6]:
treemap_df = con.execute("""
SELECT a.Industry, a.Segment,
    SUM(o.Amount) FILTER (WHERE o.Stage = '06 - Closed Won') AS won_revenue
FROM Opportunities o
JOIN Accounts a USING (AccountId)
GROUP BY 1, 2
HAVING SUM(o.Amount) FILTER (WHERE o.Stage = '06 - Closed Won') > 0
""").df()

fig = px.treemap(treemap_df, path=['Industry', 'Segment'], values='won_revenue',
                 color='won_revenue', color_continuous_scale='Blues',
                 height=520,
                 title='Won Revenue by Industry → Segment')
fig.show()

### 1.5 Rep quota attainment

In [7]:
QUOTA = 5_000_000

rep = con.execute("""
SELECT u.OwnerName AS rep, u.Region,
    SUM(o.Amount) FILTER (WHERE o.Stage = '06 - Closed Won') AS won_revenue
FROM Opportunities o
JOIN Users u USING (OwnerId)
WHERE o.CloseDate >= '2025-06-01'
GROUP BY 1, 2
ORDER BY won_revenue DESC NULLS LAST
""").df()
rep['attainment'] = rep['won_revenue'] / QUOTA

fig = px.bar(rep, x='attainment', y='rep', orientation='h',
             color='Region', height=440,
             title='Rep Quota Attainment (annual quota = $5M)',
             labels={'attainment': 'Attainment vs $5M Quota', 'rep': ''})
fig.add_vline(x=1.0, line=dict(color=DANGER, dash='dash', width=2),
              annotation_text='100% quota', annotation_position='top')
fig.update_xaxes(tickformat='.0%')
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

### Design Notes — Sales Performance Overview

This view answers one question and one question only: *are we on track this quarter, and where is the variance coming from?*

- **Won Revenue and Win Rate sit at the top** because they are the lagging outputs the CRO is measured on. Everything else on this page is in service of explaining variance in those two numbers.
- **The Region × Quarter heatmap is the strongest chart on this view.** A revenue trend line can hide a region that's collapsing if another region is growing fast enough to compensate. The heatmap exposes that.
- **The treemap is the next-best-bet chart.** Bigger rectangles tell you where today's revenue comes from; the color gradient hints at where additional investment would be defensible.
- **Forecast is deliberately excluded.** Forecast belongs on Pipeline Health (§2), where a Sales Manager can act on a specific stuck deal. On an executive view, forecast becomes a number to debate, not a lever to pull.


---

# §2. Pipeline Health

**The question:** *Where is pipeline stuck, and who needs help?*

**Audience:** Sales Manager / Front-line ops

This is the working surface a manager opens to identify coaching moments. The funnel + conversion table together expose stage-to-stage leak. The boxplot converts a vague "deals are slow" complaint into a measurable distribution.


### 2.1 Stage funnel

In [8]:
funnel = con.execute("""
SELECT Stage, COUNT(*) AS opp_count, SUM(Amount) AS total_amount
FROM Opportunities
WHERE Stage NOT LIKE '07%'
GROUP BY Stage ORDER BY Stage
""").df()

fig = go.Figure(go.Funnel(y=funnel['Stage'], x=funnel['opp_count'],
                          textinfo='value+percent initial',
                          marker={'color': PRIMARY}))
fig.update_layout(title='Opportunity Funnel (open + Won)', height=460)
fig.show()

### 2.2 Stage duration boxplot

In [9]:
dur = con.execute("""
SELECT Stage,
    CASE
        WHEN Stage IN ('06 - Closed Won','07 - Closed Lost')
            THEN DATE_DIFF('day', CreatedDate, CloseDate)
        ELSE DATE_DIFF('day', CreatedDate, CURRENT_DATE)
    END AS days_open
FROM Opportunities
""").df()

fig = px.box(dur, x='Stage', y='days_open', color='Stage', height=460,
             title='Stage Duration Distribution (days from opportunity creation)')
fig.update_layout(showlegend=False)
fig.update_yaxes(title='Days')
fig.show()

### 2.3 Stuck deals (>60 days open)

In [10]:
stuck = con.execute("""
SELECT o.OppId, a.AccountName AS account_name, o.Stage, o.Amount,
    DATE_DIFF('day', o.CreatedDate, CURRENT_DATE) AS days_open,
    u.OwnerName AS rep_name
FROM Opportunities o
JOIN Accounts a USING (AccountId)
JOIN Users u    USING (OwnerId)
WHERE o.Stage NOT IN ('06 - Closed Won','07 - Closed Lost')
  AND DATE_DIFF('day', o.CreatedDate, CURRENT_DATE) > 60
ORDER BY o.Amount DESC LIMIT 15
""").df()

stuck['Amount'] = stuck['Amount'].apply(lambda x: f'${x:,.0f}')
stuck

,OppId,account_name,Stage,Amount,days_open,rep_name
0,O000371,Account-00222,04 - Proposal,"$797,300",449,Sky Rossi
1,O000273,Account-00162,02 - Qualification,"$795,700",1132,Riley Chen
2,O000927,Account-00550,02 - Qualification,"$786,000",1212,Reese Müller
3,O000914,Account-00543,04 - Proposal,"$777,100",196,Casey Garcia
4,O000958,Account-00570,02 - Qualification,"$771,200",436,Avery Müller
5,O001139,Account-00680,02 - Qualification,"$764,100",1144,Sam Chen
6,O000969,Account-00575,04 - Proposal,"$761,200",681,Riley Chen
7,O000841,Account-00505,05 - Negotiation,"$749,500",466,Avery Müller
8,O000951,Account-00564,05 - Negotiation,"$740,900",918,Morgan Rossi
9,O000314,Account-00187,02 - Qualification,"$729,100",774,Sam Chen


### 2.4 Rep scorecard

In [11]:
scorecard = con.execute("""
SELECT u.OwnerName AS rep, u.Region,
    COUNT(*) FILTER (WHERE o.Stage NOT LIKE '06%' AND o.Stage NOT LIKE '07%') AS open_opp_count,
    SUM(o.Amount) FILTER (WHERE o.Stage NOT LIKE '06%' AND o.Stage NOT LIKE '07%') AS open_pipeline,
    SUM(o.Amount) FILTER (WHERE o.Stage = '06 - Closed Won') AS won_revenue,
    COUNT(*) FILTER (WHERE o.Stage = '06 - Closed Won') * 1.0
        / NULLIF(COUNT(*) FILTER (WHERE o.Stage IN ('06 - Closed Won','07 - Closed Lost')), 0) AS win_rate
FROM Opportunities o
JOIN Users u USING (OwnerId)
GROUP BY 1, 2
ORDER BY won_revenue DESC NULLS LAST
""").df()

fig = px.scatter(scorecard, x='open_pipeline', y='win_rate',
                 size='won_revenue', color='Region', hover_name='rep', height=480,
                 title='Rep Scorecard — Win Rate × Open Pipeline (bubble = Won Revenue)',
                 labels={'open_pipeline':'Open Pipeline ($)', 'win_rate':'Win Rate'})
fig.update_yaxes(tickformat='.0%')
fig.show()

### Design Notes — Pipeline Health

- **The funnel + boxplot are intentionally side-by-side.** The funnel tells you *how many* deals are stuck where; the boxplot tells you *how long* deals typically wait at each stage. Manager coaching needs both — a deal "stuck at Negotiation for 90 days" only matters if the median Negotiation duration is 30 days.
- **The Stuck Deals view is a list, not a chart.** Managers act on rows, not bars. A sortable table by deal size, with rep name and days open, is what gets opened on Monday morning.
- **The Rep Scorecard uses a bubble chart, not a bar.** The bar chart of "won revenue by rep" is misleading because it conflates volume with quality. A rep with high open pipeline + low win rate is a different problem than a rep with low open pipeline + high win rate. The bubble exposes that.
- **What this view deliberately doesn't show:** activity counts (calls / emails). Activity tracking is the next layer down — a Sales Manager who cares about activity volume should drill from a rep's row into a detail view, not have it mixed into the scoreboard.


---

# §3. Account Insights

**The question:** *Which segments compound, and where is the leak?*

**Audience:** PM / GTM Strategy

This is the view I'd use to decide *where the company should make its next investment.* The cohort triangle is the most underused chart in B2B SaaS GTM — it shows whether each new vintage of accounts is monetizing faster or slower than the last, which is the earliest warning sign of PMF drift.


### 3.1 Cohort retention triangle

In [12]:
cohort = con.execute("""
SELECT
    DATE_TRUNC('month', a.CreatedDate)::DATE AS cohort_month,
    DATE_DIFF('month', DATE_TRUNC('month', a.CreatedDate)::DATE, DATE_TRUNC('month', o.CloseDate)::DATE) AS months_since_cohort,
    SUM(o.Amount) AS won_amount
FROM Accounts a
JOIN Opportunities o USING (AccountId)
WHERE o.Stage = '06 - Closed Won'
GROUP BY 1, 2
""").df()

cohort = cohort[(cohort['months_since_cohort'] >= 0) & (cohort['months_since_cohort'] <= 18)]
pivot = cohort.pivot(index='cohort_month', columns='months_since_cohort', values='won_amount').fillna(0)

fig = px.imshow(pivot, color_continuous_scale='Blues', aspect='auto', height=520,
                labels=dict(color='Won Revenue', x='Months Since Cohort', y='Cohort Month'),
                title='Cohort Triangle: Won Revenue by Acquisition Cohort × Months Since')
fig.show()

### 3.2 ARPA matrix by Segment × Industry

In [13]:
arpa = con.execute("""
SELECT a.Segment, a.Industry,
    COUNT(DISTINCT CASE WHEN o.Stage = '06 - Closed Won' THEN a.AccountId END) AS won_account_count,
    SUM(CASE WHEN o.Stage = '06 - Closed Won' THEN o.Amount ELSE 0 END)
        / NULLIF(COUNT(DISTINCT CASE WHEN o.Stage = '06 - Closed Won' THEN a.AccountId END), 0) AS arpa
FROM Accounts a
LEFT JOIN Opportunities o USING (AccountId)
GROUP BY 1, 2
HAVING won_account_count > 0
""").df()

pivot = arpa.pivot(index='Industry', columns='Segment', values='arpa').fillna(0)

fig = px.imshow(pivot, color_continuous_scale='Teal', aspect='auto', height=460,
                labels=dict(color='ARPA ($)'),
                title='ARPA by Segment × Industry — the wedge view')
fig.update_traces(text=pivot.round(0).astype(int).astype(str), texttemplate='%{text}')
fig.show()

### 3.3 Loss reason Pareto

In [14]:
loss = con.execute("""
SELECT LossReason, COUNT(*) AS lost_count
FROM Opportunities WHERE Stage = '07 - Closed Lost'
GROUP BY 1 ORDER BY lost_count DESC
""").df()
loss['cum_pct'] = (loss['lost_count'].cumsum() / loss['lost_count'].sum()) * 100

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=loss['LossReason'], y=loss['lost_count'],
                     marker_color=PRIMARY, name='Lost deals'), secondary_y=False)
fig.add_trace(go.Scatter(x=loss['LossReason'], y=loss['cum_pct'],
                         mode='lines+markers', name='Cumulative %',
                         line=dict(color=DANGER, width=3)), secondary_y=True)
fig.update_layout(title='Loss Reason Pareto', height=480, hovermode='x unified')
fig.update_yaxes(title_text='Lost Deal Count', secondary_y=False)
fig.update_yaxes(title_text='Cumulative %', secondary_y=True, ticksuffix='%')
fig.show()

### 3.4 Win rate by Industry

In [15]:
wr = con.execute("""
SELECT a.Industry,
    COUNT(DISTINCT CASE WHEN o.Stage = '06 - Closed Won' THEN o.OppId END) * 1.0
        / NULLIF(COUNT(DISTINCT CASE WHEN o.Stage IN ('06 - Closed Won','07 - Closed Lost') THEN o.OppId END), 0) AS win_rate,
    COUNT(DISTINCT CASE WHEN o.Stage IN ('06 - Closed Won','07 - Closed Lost') THEN o.OppId END) AS closed_count
FROM Accounts a
JOIN Opportunities o USING (AccountId)
GROUP BY 1 HAVING closed_count >= 5
ORDER BY win_rate DESC
""").df()
mean_wr = (wr['win_rate'] * wr['closed_count']).sum() / wr['closed_count'].sum()

fig = px.bar(wr, x='Industry', y='win_rate',
             color='win_rate', color_continuous_scale='Blues',
             title=f'Win Rate by Industry (overall avg = {mean_wr*100:.1f}%)', height=440)
fig.add_hline(y=mean_wr, line=dict(color=DANGER, dash='dash', width=2),
              annotation_text=f'Average {mean_wr*100:.1f}%', annotation_position='top right')
fig.update_yaxes(tickformat='.0%', title='Win Rate')
fig.update_layout(coloraxis_showscale=False)
fig.show()

### Design Notes — Account Insights

- **The cohort triangle is the most underused chart in B2B SaaS GTM.** Most companies look at MRR / ARR by month — that's a lagging indicator that mixes new acquisition with cohort decay. The triangle separates them: each row is one acquisition vintage, and you can see whether month-3 revenue per cohort is rising or falling over time. Falling = PMF or pricing drift.
- **The ARPA matrix is where I look for the next strategic bet.** An Industry × Segment cell with **high ARPA but low account count** is a wedge worth investing in — the unit economics work, the only constraint is acquisition. Loop with marketing and double down.
- **The loss Pareto closes the analytical loop.** If "Price" dominates the losses → packaging is broken; if "Product Fit" dominates → the issue is upstream of sales (product or marketing positioning); if "No Decision" dominates → sales process discipline is the gap.
- **Win rate by Industry plus an average line** is intentional: the line tells you which industries are above water and which are dragging the overall number down. Without the average line, the bar chart just looks like a ranking.


---

# §4. Customer Health

**The question:** *Which customers will churn or expand next?*

**Audience:** CS Manager / CSM Lead

This is the CS Manager's working surface — not the exec one. The four KPIs at the top are the four numbers a CS leader can be measured on in a single quarter. The Renewal Risk Watchlist is intentionally narrow: it's the queue a CSM needs on Monday morning, not a database of all customers.


### 4.1 KPI block

In [16]:
cs_kpi = con.execute("""
SELECT
    SUM(CASE WHEN Status = 'Active'  THEN ARR ELSE 0 END) AS active_arr,
    1.0 - SUM(CASE WHEN Status = 'Churned' THEN ARR ELSE 0 END)
        / NULLIF(SUM(ARR), 0) AS gross_retention,
    (SELECT AVG(NPS) FROM HealthScores) AS avg_nps,
    (SELECT COUNT(*) FROM HealthScores WHERE HealthBand = 'Red') AS red_count
FROM Subscriptions
""").df()

print(f"  Active ARR:        ${cs_kpi.iloc[0]['active_arr']:>14,.0f}")
print(f"  Gross Retention:   {cs_kpi.iloc[0]['gross_retention']*100:>14.1f}%")
print(f"  Avg NPS:           {cs_kpi.iloc[0]['avg_nps']:>14.1f}")
print(f"  Red accounts:      {int(cs_kpi.iloc[0]['red_count']):>14d}")

  Active ARR:        $    50,145,600
  Gross Retention:             89.1%
  Avg NPS:                     22.5
  Red accounts:                  33


### 4.2 Health band distribution

In [17]:
hb = con.execute("""
SELECT HealthBand, COUNT(*) AS account_count, SUM(s.ARR) AS total_arr
FROM Subscriptions s
JOIN HealthScores  h USING (SubscriptionId)
GROUP BY HealthBand
""").df()

color_map = {'Green': '#00C2A8', 'Yellow': '#F5A623', 'Red': '#E94E1B'}

fig = px.pie(hb, names='HealthBand', values='account_count',
             color='HealthBand', color_discrete_map=color_map,
             hole=0.55, height=400,
             title='Customer Distribution by Health Band')
fig.show()

### 4.3 NPS distribution by health band

In [18]:
nps = con.execute("""SELECT HealthBand, NPS FROM HealthScores""").df()

fig = px.histogram(nps, x='NPS', color='HealthBand',
                   color_discrete_map=color_map,
                   nbins=20, barmode='stack', height=420,
                   title='NPS Distribution by Health Band')
fig.show()

### 4.4 MAU ratio trend by Plan

In [19]:
mau = con.execute("""
SELECT s.Plan, DATE_TRUNC('month', u.Month)::DATE AS month,
    AVG(u.MAURatio) AS avg_mau_ratio
FROM UsageMetrics u
JOIN Subscriptions s USING (SubscriptionId)
WHERE u.Month >= '2025-06-01'
GROUP BY 1, 2 ORDER BY 2, 1
""").df()

fig = px.line(mau, x='month', y='avg_mau_ratio', color='Plan',
              markers=True, height=440,
              title='Monthly Average MAU Ratio by Plan',
              labels={'avg_mau_ratio':'MAU / Seats (avg)', 'month':'Month'})
fig.update_yaxes(tickformat='.0%')
fig.update_layout(hovermode='x unified')
fig.show()

### 4.5 Feature adoption matrix

In [20]:
adopt = con.execute("""
WITH latest AS (
    SELECT SubscriptionId, AVG(FeatureAdoptionRate) AS adoption
    FROM UsageMetrics WHERE Month >= '2026-02-01' GROUP BY 1
)
SELECT s.Plan,
    CASE WHEN l.adoption < 0.4 THEN 'Low'
         WHEN l.adoption < 0.7 THEN 'Mid'
         ELSE 'High' END AS adoption_band,
    COUNT(*) AS subscription_count
FROM Subscriptions s
JOIN latest l USING (SubscriptionId)
GROUP BY 1, 2
""").df()

pivot = adopt.pivot(index='Plan', columns='adoption_band', values='subscription_count').fillna(0)
pivot = pivot[['Low', 'Mid', 'High']]

fig = px.imshow(pivot, color_continuous_scale='Teal',
                text_auto=True, aspect='auto', height=380,
                labels=dict(color='Subscription Count'),
                title='Feature Adoption Matrix — Plan × Adoption Band')
fig.show()

### 4.6 Renewal Risk Watchlist

In [21]:
watchlist = con.execute("""
SELECT a.AccountName, s.Plan, s.ARR,
    h.HealthBand, h.HealthScore, h.NPS,
    s.RenewalDate, DATE_DIFF('day', CURRENT_DATE, s.RenewalDate) AS days_to_renewal,
    h.CSMOwner
FROM Subscriptions s
JOIN Accounts      a USING (AccountId)
JOIN HealthScores  h USING (SubscriptionId)
WHERE s.Status <> 'Churned'
  AND DATE_DIFF('day', CURRENT_DATE, s.RenewalDate) BETWEEN 0 AND 90
  AND h.HealthBand IN ('Red', 'Yellow')
ORDER BY s.ARR DESC LIMIT 25
""").df()
watchlist['ARR'] = watchlist['ARR'].apply(lambda x: f'${x:,.0f}')
watchlist

,AccountName,Plan,ARR,HealthBand,HealthScore,NPS,RenewalDate,days_to_renewal,CSMOwner
0,Account-00406,Enterprise,"$1,323,900",Yellow,74,77,2026-06-22,36,CSM-01
1,Account-00584,Business,"$218,900",Yellow,60,16,2026-06-02,16,CSM-04
2,Account-00085,Business,"$182,500",Yellow,65,54,2026-05-22,5,CSM-07
3,Account-00673,Business,"$170,800",Yellow,62,60,2026-08-07,82,CSM-02
4,Account-00535,Growth,"$106,000",Yellow,71,39,2026-05-31,14,CSM-05
5,Account-00200,Growth,"$92,600",Yellow,47,66,2026-06-13,27,CSM-05
6,Account-00666,Growth,"$78,300",Yellow,64,57,2026-08-14,89,CSM-01
7,Account-00477,Growth,"$42,300",Yellow,64,64,2026-06-02,16,CSM-08
8,Account-00706,Growth,"$33,900",Yellow,63,-3,2026-08-09,84,CSM-08
9,Account-00333,Growth,"$26,000",Yellow,64,-1,2026-06-26,40,CSM-07


### Design Notes — Customer Health

- **The KPIs at the top are the four numbers a CS leader can be measured on.** Active ARR (the size of the book), Gross Retention (the leak), avg NPS (the sentiment), Red accounts (the immediate workload). Add more and the page becomes a database; remove any and the page becomes a vanity dashboard.
- **The MAU ratio trend by cohort is the most useful leading indicator on this page.** Health scores are *outputs* — they reflect what already happened. A cohort whose MAU ratio is sliding three months in a row is at risk regardless of what the health score says today. This is where the CS team can intervene *before* the score turns red.
- **The Renewal Risk Watchlist is intentionally narrow** — Red or Yellow band, renewal in the next 90 days, sorted by ARR. That's the queue a CSM actually needs on Monday morning. A wider filter ("all customers", "all renewals this year") turns it into a database that nobody opens.
- **What this view deliberately excludes:** logo retention. Logo retention (count-based) is misleading when ARR is heavily concentrated — losing one 7-figure customer matters more than losing five 5-figure ones. Gross retention (ARR-weighted) is the truer measure for a CS leader.


---

## What's next

This is **Phase 1 of 5** in the broader portfolio. Planned subsequent phases:

- **Phase 3 — EC Marketplace:** Same framework tested on a transactional high-volume business
- **Phase 4 — Advanced Analytics with pandas:** cohort LTV, lead scoring (logistic regression on activities), pipeline anomaly detection
- **Phase 5 — Field Sales Optimization:** geographic visualization of merchant density × GMV potential, Voronoi territory partitioning, visit-vs-call hybrid scoring

---

**[github.com/dkamehat](https://github.com/dkamehat)**
